In [1]:
# CELL 1
import os, time, json, warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import joblib

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split

from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.svm import SVC, OneClassSVM
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier

from xgboost import XGBClassifier

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score

# deep learning
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping

# shap & plotting
import shap
import matplotlib.pyplot as plt
import seaborn as sns

# ---------- PATHS ----------
BASE = "/Users/adityaray/Desktop/project/backend/models"
DATA_PATH = "/Users/adityaray/Desktop/project/dataset/kdd_dataset.csv"  # edit if needed

print("BASE:", BASE)
print("DATA_PATH:", DATA_PATH)

BASE: /Users/adityaray/Desktop/project/backend/models
DATA_PATH: /Users/adityaray/Desktop/project/dataset/kdd_dataset.csv


In [2]:
# CELL 2
os.makedirs(f"{BASE}/supervised", exist_ok=True)
os.makedirs(f"{BASE}/unsupervised", exist_ok=True)
os.makedirs(f"{BASE}/preprocessing", exist_ok=True)
os.makedirs(f"{BASE}/shap", exist_ok=True)
os.makedirs(f"{BASE}/logs", exist_ok=True)
print("Folders ready.")

Folders ready.


In [3]:
# CELL 3
df = pd.read_csv(DATA_PATH)
print("Raw shape:", df.shape)
# Drop unnamed index column(s) if exist (common when CSV saved with index)
unnamed = [c for c in df.columns if str(c).startswith("Unnamed") or str(c).strip()==""]
if unnamed:
    print("Dropping unnamed columns:", unnamed)
    df = df.drop(columns=unnamed)
print("Cleaned shape:", df.shape)
display(df.head())

Raw shape: (148517, 44)
Dropping unnamed columns: ['Unnamed: 0']
Cleaned shape: (148517, 43)


,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,...,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,class,difficulty
0,0,tcp,ftp_data,SF,491,0,0,0,0,0,...,0.17,0.03,0.17,0.00,0.00,0.00,0.05,0.00,normal,20
1,0,udp,other,SF,146,0,0,0,0,0,...,0.00,0.60,0.88,0.00,0.00,0.00,0.00,0.00,normal,15
2,0,tcp,private,S0,0,0,0,0,0,0,...,0.10,0.05,0.00,0.00,1.00,1.00,0.00,0.00,neptune,19
3,0,tcp,http,SF,232,8153,0,0,0,0,...,1.00,0.00,0.03,0.04,0.03,0.01,0.00,0.01,normal,21
4,0,tcp,http,SF,199,420,0,0,0,0,...,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,normal,21


In [4]:
# CELL 4
expected = ["duration","protocol_type","service","flag","src_bytes","dst_bytes","land",
            "wrong_fragment","urgent","hot","num_failed_logins","logged_in","num_compromised",
            "root_shell","su_attempted","num_root","num_file_creations","num_shells",
            "num_access_files","num_outbound_cmds","is_host_login","is_guest_login","count",
            "srv_count","serror_rate","srv_serror_rate","rerror_rate","srv_rerror_rate",
            "same_srv_rate","diff_srv_rate","srv_diff_host_rate","dst_host_count","dst_host_srv_count",
            "dst_host_same_srv_rate","dst_host_diff_srv_rate","dst_host_same_src_port_rate",
            "dst_host_srv_diff_host_rate","dst_host_serror_rate","dst_host_srv_serror_rate",
            "dst_host_rerror_rate","dst_host_srv_rerror_rate","class","difficulty"]

missing = [c for c in expected if c not in df.columns]
if missing:
    raise ValueError(f"Missing expected columns: {missing}")

# drop difficulty (not used)
if "difficulty" in df.columns:
    df = df.drop(columns=["difficulty"])
print("Columns OK. Shape now:", df.shape)

Columns OK. Shape now: (148517, 42)


In [5]:
# CELL 5
df["binary_label"] = df["class"].apply(lambda x: 0 if x == "normal" else 1)
print("Binary label distribution:")
print(df["binary_label"].value_counts())

Binary label distribution:
binary_label
0    77054
1    71463
Name: count, dtype: int64


In [6]:
# CELL 6
df_attack_mapping = df[["class","binary_label"]].copy()
df_attack_mapping.to_csv(f"{BASE}/preprocessing/attack_mapping.csv", index=False)

attack_frequency = df[df["binary_label"]==1]["class"].value_counts().to_dict()
joblib.dump(attack_frequency, f"{BASE}/preprocessing/attack_frequency.pkl")
print("Saved attack mapping & frequency.")

Saved attack mapping & frequency.


In [7]:
# CELL 7
categorical_cols = ["protocol_type","service","flag"]
X = df.drop(columns=["class","binary_label"])
y = df["binary_label"]
numeric_cols = [c for c in X.columns if c not in categorical_cols]
print("Numeric cols:", len(numeric_cols), "Categorical cols:", categorical_cols)

Numeric cols: 38 Categorical cols: ['protocol_type', 'service', 'flag']


In [8]:
# CELL 8
encoder = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
X_cat = encoder.fit_transform(X[categorical_cols])
joblib.dump(encoder, f"{BASE}/preprocessing/encoder.pkl")
print("Encoder saved. Encoded cat shape:", X_cat.shape)

Encoder saved. Encoded cat shape: (148517, 84)


In [9]:
# CELL 9
scaler = StandardScaler()
X_num = scaler.fit_transform(X[numeric_cols])
joblib.dump(scaler, f"{BASE}/preprocessing/scaler.pkl")
print("Scaler saved. Numeric shape:", X_num.shape)

Scaler saved. Numeric shape: (148517, 38)


In [10]:
# CELL 10
X_final = np.hstack([X_num, X_cat])
# create encoded category feature names
cat_feature_names = list(encoder.get_feature_names_out(categorical_cols))
all_feature_names = numeric_cols + cat_feature_names
joblib.dump(all_feature_names, f"{BASE}/preprocessing/selected_features.pkl")
print("X_final shape:", X_final.shape, "Total features:", len(all_feature_names))

X_final shape: (148517, 122) Total features: 122


In [11]:
# CELL 11
X_train, X_test, y_train, y_test = train_test_split(
    X_final, y, test_size=0.2, random_state=42, shuffle=True)
print("Train:", X_train.shape, "Test:", X_test.shape)

Train: (118813, 122) Test: (29704, 122)


In [12]:
# CELL 12
def save_model(obj, path):
    if isinstance(obj, (type(None))):
        return
    if path.endswith(".pkl"):
        joblib.dump(obj, path)
    else:
        try:
            obj.save(path)
        except:
            joblib.dump(obj, path)

def time_and_train(name, model, Xtr, ytr, path):
    t0 = time.time()
    model.fit(Xtr, ytr)
    dt = time.time() - t0
    save_model(model, path)
    print(f"{name} done in {dt:.2f}s -> {path}")
    return model, dt

In [13]:
# CELL 13
models_info = {}

rf, t = time_and_train("RandomForest", RandomForestClassifier(n_jobs=-1, random_state=42),
                       X_train, y_train, f"{BASE}/supervised/random_forest.pkl")
models_info["RandomForest"] = (rf,t)

xgb, t = time_and_train("XGBoost", XGBClassifier(use_label_encoder=False, eval_metric="logloss", n_jobs=-1, random_state=42),
                        X_train, y_train, f"{BASE}/supervised/xgboost.pkl")
models_info["XGBoost"] = (xgb,t)

svm, t = time_and_train("SVM", SVC(probability=True, kernel="rbf", random_state=42),
                        X_train, y_train, f"{BASE}/supervised/svm.pkl")
models_info["SVM"] = (svm,t)

lr, t = time_and_train("LogisticRegression", LogisticRegression(max_iter=5000),
                      X_train, y_train, f"{BASE}/supervised/logistic_regression.pkl")
models_info["LogisticRegression"] = (lr,t)

knn, t = time_and_train("KNN", KNeighborsClassifier(), X_train, y_train, f"{BASE}/supervised/knn.pkl")
models_info["KNN"] = (knn,t)

nb, t = time_and_train("NaiveBayes", GaussianNB(), X_train, y_train, f"{BASE}/supervised/naive_bayes.pkl")
models_info["NaiveBayes"] = (nb,t)

dt, t = time_and_train("DecisionTree", DecisionTreeClassifier(random_state=42), X_train, y_train, f"{BASE}/supervised/decision_tree.pkl")
models_info["DecisionTree"] = (dt,t)

print("Supervised training complete.")

RandomForest done in 1.27s -> /Users/adityaray/Desktop/project/backend/models/supervised/random_forest.pkl
XGBoost done in 5.85s -> /Users/adityaray/Desktop/project/backend/models/supervised/xgboost.pkl
SVM done in 513.94s -> /Users/adityaray/Desktop/project/backend/models/supervised/svm.pkl
LogisticRegression done in 1.41s -> /Users/adityaray/Desktop/project/backend/models/supervised/logistic_regression.pkl
KNN done in 0.01s -> /Users/adityaray/Desktop/project/backend/models/supervised/knn.pkl
NaiveBayes done in 0.09s -> /Users/adityaray/Desktop/project/backend/models/supervised/naive_bayes.pkl
DecisionTree done in 0.97s -> /Users/adityaray/Desktop/project/backend/models/supervised/decision_tree.pkl
Supervised training complete.


In [14]:
# CELL 14
results = []
for name,(model, ttime) in models_info.items():
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    try:
        prob = model.predict_proba(X_test)[:,1]
        roc = roc_auc_score(y_test, prob)
    except:
        roc = None
    results.append({"Model":name, "TrainTime(s)":round(ttime,2),
                    "Accuracy":round(acc,4),"Precision":round(prec,4),
                    "Recall":round(rec,4),"F1":round(f1,4),"ROC_AUC":round(roc,4) if roc is not None else None})
results_df = pd.DataFrame(results).sort_values("F1", ascending=False)
results_df.to_csv(f"{BASE}/supervised/model_performance.csv", index=False)
display(results_df)
print("Saved model_performance.csv")

,Model,TrainTime(s),Accuracy,Precision,Recall,F1,ROC_AUC
1,XGBoost,5.85,0.9965,0.9966,0.9960,0.9963,0.9999
0,RandomForest,1.27,0.9958,0.9970,0.9942,0.9956,0.9998
6,DecisionTree,0.97,0.9949,0.9955,0.9939,0.9947,0.9952
4,KNN,0.01,0.9913,0.9911,0.9908,0.9909,0.9981
2,SVM,513.94,0.9847,0.9869,0.9812,0.9841,0.9982
3,LogisticRegression,1.41,0.9570,0.9668,0.9428,0.9546,0.9912
5,NaiveBayes,0.09,0.8172,0.9935,0.6231,0.7659,0.9744


Saved model_performance.csv


In [15]:
# CELL 15
for name in ["RandomForest","XGBoost"]:
    model = models_info[name][0]
    cm = confusion_matrix(y_test, model.predict(X_test))
    plt.figure(figsize=(4,3))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
    plt.title(f"{name} Confusion Matrix")
    p = f"{BASE}/shap/{name}_confusion_matrix.png"
    plt.savefig(p, bbox_inches="tight", dpi=150)
    plt.close()
    print("Saved", p)

Saved /Users/adityaray/Desktop/project/backend/models/shap/RandomForest_confusion_matrix.png
Saved /Users/adityaray/Desktop/project/backend/models/shap/XGBoost_confusion_matrix.png


In [16]:
# CELL 16
normal_data = X_final[y==0]
print("Normal data shape:", normal_data.shape)

iso = IsolationForest(contamination=0.1, random_state=42)
t0=time.time(); iso.fit(normal_data); joblib.dump(iso, f"{BASE}/unsupervised/isolation_forest.pkl")
print("IsolationForest trained in", round(time.time()-t0,2),"s")

ocsvm = OneClassSVM(kernel="rbf", gamma="scale")
t0=time.time(); ocsvm.fit(normal_data); joblib.dump(ocsvm, f"{BASE}/unsupervised/one_class_svm.pkl")
print("OneClassSVM trained in", round(time.time()-t0,2),"s")

Normal data shape: (77054, 122)
IsolationForest trained in 0.24 s
OneClassSVM trained in 327.3 s


In [17]:
# CELL 17
input_dim = X_final.shape[1]
print("Autoencoder input dim:", input_dim)

autoencoder = Sequential([
    Dense(64, activation='relu', input_dim=input_dim),
    Dense(32, activation='relu'),
    Dense(16, activation='relu'),
    Dense(32, activation='relu'),
    Dense(64, activation='relu'),
    Dense(input_dim, activation='sigmoid')
])
autoencoder.compile(optimizer='adam', loss='mse')
es = EarlyStopping(monitor="loss", patience=3, restore_best_weights=True)

t0=time.time()
autoencoder.fit(normal_data, normal_data, epochs=50, batch_size=256, callbacks=[es], verbose=1)
dt=time.time()-t0
autoencoder.save(f"{BASE}/unsupervised/autoencoder.h5")
print("Autoencoder trained and saved in", round(dt,2),"s")

Autoencoder input dim: 122
Epoch 1/50


2025-11-15 20:12:49.747401: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M4
2025-11-15 20:12:49.747449: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 16.00 GB
2025-11-15 20:12:49.747456: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 5.92 GB
2025-11-15 20:12:49.747467: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2025-11-15 20:12:49.747476: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)
2025-11-15 20:12:50.193277: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


301/301 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - loss: 0.2152
Epoch 2/50
301/301 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - loss: 0.2017
Epoch 3/50
301/301 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - loss: 0.3818
Epoch 4/50
301/301 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - loss: 0.4333
Epoch 5/50
301/301 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - loss: 0.4282


Autoencoder trained and saved in 12.84 s


In [18]:
# CELL 18
recon = autoencoder.predict(normal_data)
mse = np.mean(np.square(normal_data - recon), axis=1)
plt.figure(figsize=(6,3)); plt.hist(mse,bins=80); plt.title("Reconstruction error (normal)"); plt.xlabel("MSE")
plt.savefig(f"{BASE}/shap/reconstruction_error_hist.png", bbox_inches="tight", dpi=150); plt.close()
mse_mean, mse_std = float(mse.mean()), float(mse.std())
ae_threshold = mse_mean + 3*mse_std
iso_scores = iso.decision_function(normal_data)
iso_threshold = float(np.percentile(iso_scores,5))
thresholds = {"ae_mse_mean":mse_mean,"ae_mse_std":mse_std,"ae_threshold":ae_threshold,"iso_threshold":iso_threshold}
joblib.dump(thresholds, f"{BASE}/unsupervised/thresholds.pkl")
print("Saved thresholds:", thresholds)

2408/2408 ━━━━━━━━━━━━━━━━━━━━ 2s 988us/step
Saved thresholds: {'ae_mse_mean': 0.3173971870405504, 'ae_mse_std': 6.891309640399976, 'ae_threshold': 20.991326108240482, 'iso_threshold': -0.025914826531149404}


In [19]:
# CELL 19
# sample for SHAP to keep runtime reasonable
shap_sample = min(2000, X_train.shape[0])
idx = np.random.choice(range(X_train.shape[0]), shap_sample, replace=False)
X_shap = X_train[idx]
feature_names = joblib.load(f"{BASE}/preprocessing/selected_features.pkl")
X_shap_df = pd.DataFrame(X_shap, columns=feature_names)

# RandomForest SHAP
explainer_rf = shap.TreeExplainer(models_info["RandomForest"][0])
shap_values_rf = explainer_rf.shap_values(X_shap_df) if hasattr(explainer_rf, "shap_values") else explainer_rf(X_shap_df)
plt.figure(figsize=(8,6)); shap.summary_plot(shap_values_rf, X_shap_df, show=False); plt.title("SHAP summary - RF")
plt.savefig(f"{BASE}/shap/randomforest_shap_summary.png", bbox_inches="tight", dpi=150); plt.close()
print("Saved RandomForest SHAP summary")

# XGBoost SHAP
explainer_xgb = shap.Explainer(models_info["XGBoost"][0], X_shap_df)
shap_values_xgb = explainer_xgb(X_shap_df)
plt.figure(figsize=(8,6)); shap.summary_plot(shap_values_xgb, X_shap_df, show=False); plt.title("SHAP summary - XGB")
plt.savefig(f"{BASE}/shap/xgboost_shap_summary.png", bbox_inches="tight", dpi=150); plt.close()
print("Saved XGBoost SHAP summary")

# save one local waterfall/force-style for RF (matplotlib-friendly)
sample_row = X_shap_df.iloc[[0]]
sv = explainer_rf.shap_values(sample_row) if hasattr(explainer_rf, "shap_values") else explainer_rf(sample_row)
try:
    plt.figure(figsize=(6,3))
    shap.plots.waterfall(shap.Explanation(values=sv, base_values=explainer_rf.expected_value, data=sample_row), show=False)
    plt.savefig(f"{BASE}/shap/randomforest_shap_waterfall_sample.png", bbox_inches="tight", dpi=150)
    plt.close()
    print("Saved RF local SHAP waterfall")
except Exception as e:
    print("Could not save waterfall plot:", e)

Saved RandomForest SHAP summary
Saved XGBoost SHAP summary
Could not save waterfall plot: The waterfall plot can currently only plot a single explanation, but a matrix of explanations (shape (1, 122, 2)) was passed! Perhaps try `shap.plots.waterfall(shap_values[0])` or for multi-output models, try `shap.plots.waterfall(shap_values[0, 0])`.


<Figure size 800x600 with 0 Axes>

<Figure size 600x300 with 0 Axes>

In [20]:
# CELL 20
summary = {
    "base": BASE,
    "data_path": DATA_PATH,
    "supervised": os.listdir(f"{BASE}/supervised"),
    "unsupervised": os.listdir(f"{BASE}/unsupervised"),
    "preprocessing": os.listdir(f"{BASE}/preprocessing"),
    "shap": os.listdir(f"{BASE}/shap"),
    "train_samples": int(X_train.shape[0]),
    "test_samples": int(X_test.shape[0]),
    "thresholds": thresholds
}
with open(f"{BASE}/logs/artifacts_summary.json","w") as f:
    json.dump(summary, f, indent=2)
print("Artifacts summary saved:", f"{BASE}/logs/artifacts_summary.json")
# print a small tree
for root,_,files in os.walk(BASE):
    print(root, "->", len(files), "files")

Artifacts summary saved: /Users/adityaray/Desktop/project/backend/models/logs/artifacts_summary.json
/Users/adityaray/Desktop/project/backend/models -> 1 files
/Users/adityaray/Desktop/project/backend/models/unsupervised -> 4 files
/Users/adityaray/Desktop/project/backend/models/shap -> 5 files
/Users/adityaray/Desktop/project/backend/models/supervised -> 8 files
/Users/adityaray/Desktop/project/backend/models/logs -> 1 files
/Users/adityaray/Desktop/project/backend/models/preprocessing -> 5 files
